# 🤖 Track A: Classical Machine Learning & Trading Strategy
**Author:** Kevin (Track A Specialist)  
**Universe:** 5 Liquid IDX Bluechip Stocks (`BBCA.JK`, `BBRI.JK`, `BMRI.JK`, `TLKM.JK`, `ASII.JK`)  
**Timeline:** 2018 – 2026  
**Core Objectives:**
1. **Classical Machine Learning Models:** Logistic Regression (Phase 7 Baseline), Random Forest (Phase 8), and XGBoost (Phase 9).
2. **Probability Calibration:** Platt Scaling (Sigmoid) and Isotonic Regression to ensure well-calibrated $P(\text{Up})$.
3. **Trading Signals & Backtesting:** Conversion to `BUY`, `HOLD`, `SELL` signals with IDX fee simulation (0.15%).
4. **Database Ingestion:** Persisting Track A probabilistic predictions into SQLite (`stock_prediction.db`).
5. **Unified Model Comparison:** Comprehensive performance analysis against Track B models and Buy & Hold benchmark.

---  
## 1. Setup & Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure project root in path
sys.path.append(os.path.abspath('..'))

from src.data.preparation import split_time_series
from src.features.technical_indicators import get_feature_columns
from src.models.classical_models import run_track_a_modeling
from src.models.backtest_track_a import run_track_a_backtesting
from src.models.save_predictions import query_prediction_summary

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("Environment initialized successfully!")

---  
## 2. Load Modeling Dataset & Chronological Split
We use the standardized 29 backward-looking technical features across the 3 strict splits:
- **Train:** 2018 – 2023 (fitting models & scalers)
- **Validation:** 2024 (probability calibration & signal thresholding)
- **Test:** 2025 – Present (strictly sterile out-of-sample evaluation)

In [ ]:
data_path = '../data/processed/modeling_data_advanced.csv'
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
feature_cols = get_feature_columns()

train_df, val_df, test_df = split_time_series(df)
print(f"Total Rows: {len(df)} | Features: {len(feature_cols)}")
print(f"Train: {len(train_df)} rows ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
print(f"Val  : {len(val_df)} rows ({val_df['date'].min().date()} to {val_df['date'].max().date()})")
print(f"Test : {len(test_df)} rows ({test_df['date'].min().date()} to {test_df['date'].max().date()})")

---  
## 3. Track A Predictive Performance on Test Set
Comparing Logistic Regression, Random Forest, and XGBoost.

In [ ]:
comp_a_path = '../reports/model_comparison_track_a.csv'
if os.path.exists(comp_a_path):
    comp_a = pd.read_csv(comp_a_path)
    display(comp_a)
else:
    comp_a = run_track_a_modeling(advanced_data_path=data_path)
    display(comp_a)

---  
## 4. Unified Comparison (Track A + Track B All Models)
Evaluating all 6 models developed across the project.

In [ ]:
comp_all_path = '../reports/model_comparison_all.csv'
if os.path.exists(comp_all_path):
    comp_all = pd.read_csv(comp_all_path)
    display(comp_all)

    # Visualizing ROC-AUC & Accuracy
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=comp_all, x='roc_auc', y='model', palette='viridis', ax=ax1)
    ax1.set_title('Test Set ROC-AUC Comparison')
    ax1.set_xlim(0.50, 0.58)

    sns.barplot(data=comp_all, x='accuracy', y='model', palette='mako', ax=ax2)
    ax2.set_title('Test Set Accuracy Comparison')
    ax2.set_xlim(0.50, 0.60)
    plt.tight_layout()
    plt.show()

---  
## 5. Realistic Trading Strategy Backtest
Simulating trading execution with 0.15% transaction costs on the sterile Test Set (2025+).

In [ ]:
bt_all_path = '../reports/trading_backtest_all.csv'
if os.path.exists(bt_all_path):
    bt_all = pd.read_csv(bt_all_path)
    display(bt_all)
    
    # Visualizing Total Return & Sharpe Ratio
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=bt_all, x='Total Return (%)', y='Model', palette='coolwarm', ax=ax1)
    ax1.axvline(0, color='red', linestyle='--', alpha=0.7)
    ax1.set_title('Strategy Total Return (%)')

    sns.barplot(data=bt_all, x='Sharpe Ratio', y='Model', palette='Blues_r', ax=ax2)
    ax2.axvline(0, color='red', linestyle='--', alpha=0.7)
    ax2.set_title('Sharpe Ratio')
    plt.tight_layout()
    plt.show()

---  
## 6. Database Verification
Inspecting the SQLite `predictions` table to ensure full persistence for Ali's dashboard API (Track C).

In [ ]:
summary = query_prediction_summary()
display(summary)